In [1]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="anhnh2002/vnTTS",
#                   repo_type="model",
#                   local_dir="model/")

In [ ]:
# !pip install cutlet

In [1]:
from pprint import pprint
import torch
import torchaudio
from tqdm import tqdm
from underthesea import sent_tokenize
from vinorm import TTSnorm
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

device = "cuda:0"

xtts_checkpoint = "model/model.pth"
xtts_config = "model/config.json"
xtts_vocab = "model/vocab.json"

config = XttsConfig()
config.load_json(xtts_config)
XTTS_MODEL = Xtts.init_from_config(config)
XTTS_MODEL.load_checkpoint(config,
                            checkpoint_path=xtts_checkpoint,
                            vocab_path=xtts_vocab,
                            use_deepspeed=False)
XTTS_MODEL.to(device)
# print(next(XTTS_MODEL.parameters()).device)

/mnt/d/Ky 4/XTTSv2-Finetuning-for-New-Languages/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get 

Xtts(
  (gpt): GPT(
    (conditioning_encoder): ConditioningEncoder(
      (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
      (attn): Sequential(
        (0): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (1): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Identity()
          (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
        )
        (2): AttentionBlock(
          (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
          (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
          (attention): QKVAttention()
          (x_proj): Ide

In [2]:
def preprocess_text(text, language="vi"):
    if language == "vi":
        text = TTSnorm(text, unknown=False, lower=False, rule=True)
    
    # split text into sentences
    if language in ["ja", "zh-cn"]:
        sentences = text.split("。")
    else:
        sentences = sent_tokenize(text)

    chunks = []
    chunk_i = ""
    len_chunk_i = 0
    for sentence in sentences:
        chunk_i += " " + sentence
        len_chunk_i += len(sentence.split())
        if len_chunk_i > 30:
            chunks.append(chunk_i.strip())
            chunk_i = ""
            len_chunk_i = 0

    if (len(chunks) > 0) and (len_chunk_i < 15):
        chunks[-1] += chunk_i
    else:
        chunks.append(chunk_i)

    return chunks

In [3]:
speaker_audio_file = "model/nu-luu-loat.wav"
# speaker_audio_file = "model/vi_man.wav"
# speaker_audio_file ="model/nu-luu-loat.wav"

gpt_cond_latent, speaker_embedding = XTTS_MODEL.get_conditioning_latents(
    audio_path=speaker_audio_file,
    gpt_cond_len=XTTS_MODEL.config.gpt_cond_len,
    max_ref_length=XTTS_MODEL.config.max_ref_len,
    sound_norm_refs=XTTS_MODEL.config.sound_norm_refs,
)

In [5]:
def tts(
    model: Xtts,
    text: str,
    language: str,
    gpt_cond_latent: torch.Tensor,
    speaker_embedding: torch.Tensor,
    verbose: bool = False,
):
    # preprocess text
    chunks = preprocess_text(text, language)

    wav_chunks = []
    for text in tqdm(chunks):
        if text.strip() == "":
            continue
        wav_chunk = model.inference(
            text=text,
            language=language,
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            length_penalty=1.0,
            repetition_penalty=10.0,
            top_k=10,
            top_p=0.5,
        )

        wav_chunk["wav"] = torch.tensor(wav_chunk["wav"])

        wav_chunks.append(wav_chunk["wav"])

    out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0).cpu()

    return out_wav

from IPython.display import Audio

audio = tts(
    model=XTTS_MODEL,
    # text="Xin chào, tôi là một hệ thống chuyển đổi văn bản tiếng Việt thành giọng nói. Hello, I am a Vietnamese text to speech conversion system.", 
    # text = """ID giáo trình: 11845
    #         Tên giáo trình: Communication and In-Group Working Skills Kỹ năng giao tiếp và cộng tác
    #         Mã chủ đề: SSG104
    #         Số tín chỉ: 3
    #         Cấp độ đào tạo: Cử nhân
    #         Phân bổ thời gian: Tổng thời lượng học là 150 giờ, bao gồm 45 giờ học trực tiếp tương đương với 60 phiên học, 0.5 giờ dành cho kỳ thi cuối kỳ, và 104.5 giờ tự học.
    #         Điều kiện tiên quyết: Không có
    # """,
    text = """🎯 Tính năng nổi trội VisionAid: Dự án là một hệ thống hỗ trợ thông minh dành cho người khiếm thị hoặc gặp khó khăn về thị lực. Ứng dụng cho phép người dùng chụp ảnh trực tiếp từ camera hoặc tải ảnh từ thiết bị lên, sau đó sử dụng công nghệ AI Image Captioning để phân tích và mô tả nội dung hình ảnh bằng văn bản một cách chi tiết, dễ hiểu. Đặc biệt, phần mô tả này sẽ được truyền qua module Text-to-Speech, giúp người dùng nghe được thông tin mô tả, cảnh báo hoặc hướng dẫn một cách tự nhiên và nhanh chóng. Tất cả quy trình diễn ra tự động, trực quan, thân thiện — hướng tới mục tiêu giúp người dùng khiếm thị tiếp cận thông tin thị giác một cách an toàn và hiệu quả hơn. Chúng tôi cam kết bảo vệ tuyệt đối mọi thông tin cá nhân và hình ảnh mà người dùng cung cấp. Khi sử dụng dịch vụ của chúng tôi, dữ liệu của bạn sẽ được xử lý trong một quy trình khép kín và tuyệt đối không được chia sẻ với bất kỳ bên thứ ba nào.""",
    language="vi",
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    verbose=True,   
)

Audio(audio, rate=24000)


  0%|          | 0/4 [00:00<?, ?it/s]

[!] Warning: The text length exceeds the character limit of 300 for language 'vi', this might cause truncated audio.


100%|██████████| 4/4 [01:50<00:00, 27.74s/it]


In [ ]:
# from llama_cpp import Llama

# import torch
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"CUDA devices: {torch.cuda.device_count()}")
# # Load mô hình
# llm = Llama(
#     model_path="ggml-vistral-7B-chat-q4_0.gguf",
#     n_ctx=2048,
#     n_threads=4,       # Tùy CPU bạn, có thể tăng nếu đa nhân
#     n_gpu_layers=32     # Nếu dùng CPU; nếu dùng GPU có thể đặt số lớn hơn 0,
# )

# # Chat loop
# print("🤖 Chatbot sẵn sàng. Gõ 'exit' để thoát.")
# while True:
#     user_input = input("👤 Bạn: ")
#     if user_input.lower() in ["exit", "quit"]:
#         print("👋 Tạm biệt!")
#         break

#     output = llm.create_completion(
#         prompt=f"[INST] {user_input} [/INST]",
#         max_tokens=512,
#         temperature=0.7,
#         top_p=0.95,
#         stop=["</s>"]
#     )
    
#     response = output["choices"][0]["text"]
#     print("🤖 Bot:", response.strip())
